# BERT: Encoder-Only Pre-training

> The GPT model we built in the previous sections uses a Decoder-Only architecture with a causal mask that restricts each token to only see the tokens before it. The original Transformer paper also describes another structure — the Encoder — which does not use a mask, allowing each token to see every position in the entire sequence.
>
> In this section we implement an Encoder-Only model called MiniBERT. By removing the causal mask, each token's field of view changes from unidirectional to bidirectional, and the pre-training task shifts from predicting the next word to filling in blanks based on context — MLM.

Cover up one word in a sentence and ask someone to guess what was hidden. For example, "I `__` you" — given only three words, a native speaker can probably guess what `__` is. That is exactly what BERT learns to do.

GPT's training revolves around generation: writing tokens one by one from left to right, never knowing what comes next. BERT does the opposite — instead of generating, it first reads the entire sentence, then answers questions about that sentence. Is the sentiment positive or negative? Which place names appear in the text? Do two sentences contradict each other? For these tasks, reading the whole thing first and then making a judgment is more straightforward than guessing word by word.

## 1. Encoder vs. Decoder

The difference between unidirectional and bidirectional can be stated in a single sentence. Given the sentence "I ate the apple", under a causal mask the model sees "I ate" without knowing whether the next word is "apple" or "phone" — it can only guess. Under bidirectional attention, the model sees "I", "ate", "the", "apple" simultaneously — every word can reference all positions in the sentence, so it knows "apple" is the object rather than the subject.

This difference determines what each model is suited for. GPT is suited for generation — each step can only see what has already been written, and the next word becomes visible only after the current one is produced. BERT is suited for understanding — it reads the entire text first, then answers any question about it.

Before diving into BERT, we need to clarify the fundamental difference between Encoder and Decoder in terms of Attention:

```
        Encoder-Only (BERT)          Decoder-Only (GPT)
        ────────────────             ────────────────

        Output: "B-PER"                Output: "France"
          ↑                               ↑
    ┌─────┴─────┐                   ┌─────┴─────┐
    │  FFN + LN  │                   │  FFN + LN  │
    ├───────────┤                   ├───────────┤
    │ Attention │                   │ Attention │
    │ (bidir!)  │                   │ (unidir!) │
    ├───────────┤                   ├───────────┤
    │  Input     │                   │  Input     │
    └───────────┘                   └───────────┘

   Each word sees ALL words         Each word sees only preceding words
   (both left and right)            (causal mask)
```

The same word "apple", under the two Attention patterns, has access to completely different amounts of context. In the Encoder, "apple" can reference "I", "ate", and every other word in the sentence. In the Decoder, "apple" can only reference words that come before it. This structural difference is the root cause of why BERT excels at understanding tasks while GPT excels at generation.

## 2. BERT's Input Representation

BERT's input is not simply a single Embedding. It is the **sum of three Embeddings**:

```
Input = Token Embedding + Segment Embedding + Position Embedding
         ↑                    ↑                    ↑
   "which word is this"   "sentence A or B"     "at which position"
```

How does this differ from GPT? GPT only has Token + Position and does not need Segment (because GPT does not handle "the relationship between two sentences").

**Why does BERT need Segment Embedding?** Because the NSP (Next Sentence Prediction) task requires judging the relationship between two sentences.

The code below demonstrates BERT's complete input construction process:

In [ ]:
# ============================================================
# Construct BERT's input from scratch, step by step
# ============================================================

# Simulate a small BERT configuration
import torch
import torch.nn as nn
torch.manual_seed(42)
VOCAB_SIZE = 100
D_MODEL = 16
MAX_LEN = 20
NUM_SEGMENTS = 2  # sentence A or sentence B

# Three Embeddings
token_embed = nn.Embedding(VOCAB_SIZE, D_MODEL)
segment_embed = nn.Embedding(NUM_SEGMENTS, D_MODEL)
position_embed = nn.Embedding(MAX_LEN, D_MODEL)

# Simulated input: two sentences
#   [CLS] I like cats [SEP] He likes dogs [SEP]
#   where [CLS]=1, [SEP]=2

sentence_A = [1, 5, 8, 3, 2]    # [CLS] I like cats [SEP]
sentence_B = [6, 8, 4, 2]        # He likes dogs [SEP]
full_ids = sentence_A + sentence_B
seq_len = len(full_ids)

print("Step 1: Token IDs")
print(f"  Input: {full_ids}")
print(f"  Meaning: [CLS] I like cats [SEP] He likes dogs [SEP]")

# Segment IDs: sentence A tokens get 0, sentence B tokens get 1
segment_ids = [0] * len(sentence_A) + [1] * len(sentence_B)
print(f"\nStep 2: Segment IDs")
print(f"  Segments: {segment_ids}")
print(f"  Meaning: first {len(sentence_A)} belong to sentence A (0), last {len(sentence_B)} belong to sentence B (1)")

# Position IDs: 0, 1, 2, ..., seq_len-1
position_ids = list(range(seq_len))
print(f"\nStep 3: Position IDs")
print(f"  Positions: {position_ids}")

# ============================================================
# Sum the three Embeddings
# ============================================================
tokens_t = torch.tensor(full_ids)
segments_t = torch.tensor(segment_ids)
positions_t = torch.tensor(position_ids)

tok_emb = token_embed(tokens_t)    # (seq_len, D_MODEL)
seg_emb = segment_embed(segments_t)  # (seq_len, D_MODEL)
pos_emb = position_embed(positions_t)  # (seq_len, D_MODEL)

input_embeddings = tok_emb + seg_emb + pos_emb

print(f"\nStep 4: Sum of three Embeddings")
print(f"  Token Embedding shape:    {tok_emb.shape}")
print(f"  Segment Embedding shape:  {seg_emb.shape}")
print(f"  Position Embedding shape: {pos_emb.shape}")
print(f"  After sum: {input_embeddings.shape}")

# Show that the same word ("like", id=8) has different embeddings
# in different sentences due to different position and segment
idx_a = 2  # "like" in sentence A
idx_b = len(sentence_A) + 1  # "like" in sentence B

print(f"\n★ Key observation: same word, different embeddings due to different position + segment")
print(f"  'like' in sentence A (pos={idx_a}, seg=0), final embedding (first 6 dims):")
print(f"    {input_embeddings[idx_a, :6].detach()}")
print(f"  'like' in sentence B (pos={idx_b}, seg=1), final embedding (first 6 dims):")
print(f"    {input_embeddings[idx_b, :6].detach()}")
print(f"  Are they the same? {(input_embeddings[idx_a] == input_embeddings[idx_b]).all().item()}")
print(f"  → Even for the same word, different position and segment produce different embeddings!")

## 3. MLM Pre-training

GPT's training task is "given the preceding words, predict the next word" (autoregressive).
BERT's training task is "cover up some words in the middle, and ask the model to guess what they are" (MLM).

```
GPT training:  I → love → you → China
               can only look → (one direction)

BERT training: I `__` you China  →  guess `__` is "love"
               look left    look right
```

BERT can see both left and right simultaneously — because its attention is **bidirectional** (no causal mask).
This makes it good at "understanding" but unable to generate — because during training it never learned to "produce words one at a time".

In [ ]:
# ============================================================
# Demonstrate BERT's bidirectional Attention (vs GPT's causal Attention)
# ============================================================
import torch
seq_len = 6

# GPT's causal mask: each position can only see itself and earlier positions
causal_mask = torch.tril(torch.ones(seq_len, seq_len))

# BERT has no mask: each position sees all positions (bidirectional)
bert_mask = torch.ones(seq_len, seq_len)

tokens = ["[CLS]", "I", "love", "you", "China", "[SEP]"]

print("=== GPT's causal Attention (unidirectional) ===")
print(f"Tokens: {tokens}")
print()
for i in range(seq_len):
    visible = [tokens[j] for j in range(seq_len) if causal_mask[i, j] == 1]
    print(f"  Position {i} ('{tokens[i]}') can see: {visible}")

print(f"\n=== BERT's Attention (bidirectional) ===")
for i in range(seq_len):
    visible = [tokens[j] for j in range(seq_len) if bert_mask[i, j] == 1]
    print(f"  Position {i} ('{tokens[i]}') can see: {visible}")

print(f"\nKey difference:")
print(f"  GPT:  'you' cannot see 'China' (not yet generated)")
print(f"  BERT: 'you' can see 'China' (whole sentence is known, bidirectional)")
print(f"  → BERT has stronger context understanding! But cannot do generation.")

## 4. MLM Training Demo

The idea behind MLM training is straightforward, but it differs fundamentally from autoregressive language model training:

- **Autoregressive models** (e.g. GPT): input token[0:n], predict token[1:n+1], and every position contributes to the loss. This is a "predict from left to right, one at a time" pattern.
- **MLM**: only randomly mask 15% of tokens. The model sees the full context (including content to the right of the masked positions), but only needs to predict the masked 15%. Loss is computed only at the masked positions.

Why does MLM only compute loss on 15% of positions? Because unmasked tokens are directly visible to the model. If we asked the model to predict all tokens without masking, it would simply learn to copy the input — it would learn nothing.

Below we build a MiniBERT and demonstrate the complete MLM training pipeline:
1. Randomly mask 15% of tokens (80% replaced with `[MASK]`, 10% replaced with a random token, 10% left unchanged)
2. The model uses bidirectional context to predict the masked tokens
3. Loss is computed only at the masked positions

In [ ]:
# ============================================================
# MiniBERT: Encoder only (bidirectional Attention, no causal mask)
# ============================================================
import torch.nn.functional as F
import math
import torch.nn as nn
import torch

class MiniBERTEncoder(nn.Module):
    """Similar Transformer Block to MiniGPT, but without causal mask (bidirectional)"""
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B, S, D = x.shape
        Q = self.W_Q(x).view(B, S, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_K(x).view(B, S, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_V(x).view(B, S, self.num_heads, self.d_k).transpose(1, 2)
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_k)
        # ★ No mask! Bidirectional!
        attn = F.softmax(scores, dim=-1)
        out = (attn @ V).transpose(1, 2).contiguous().view(B, S, D)
        return self.W_O(out)

class MiniBERTBlock(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.attention = MiniBERTEncoder(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
        )
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        x = x + self.attention(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x

class MiniBERT(nn.Module):
    def __init__(self, vocab_size, d_model=64, num_heads=4, num_layers=2, max_len=64):
        super().__init__()
        self.d_model = d_model
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        # BERT uses learned position embeddings (not sinusoidal)
        self.position_embedding = nn.Embedding(max_len, d_model)
        self.blocks = nn.ModuleList([
            MiniBERTBlock(d_model, num_heads) for _ in range(num_layers)
        ])
        self.norm_final = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)
        self.max_len = max_len

    def encode(self, x):
        """Return hidden states for each position, reusable for classification/QA tasks"""
        B, S = x.shape
        positions = torch.arange(S, device=x.device).unsqueeze(0)
        x_emb = self.token_embedding(x) + self.position_embedding(positions)
        for block in self.blocks:
            x_emb = block(x_emb)
        x_emb = self.norm_final(x_emb)
        return x_emb

    def forward(self, x):
        x_emb = self.encode(x)
        return self.lm_head(x_emb)

print("MiniBERT defined (bidirectional Attention, no causal mask)")

In [ ]:
import torch.nn.functional as F
# ============================================================
# MLM training data construction + training demo
# ============================================================
import torch
VOCAB_SIZE = 50
MASK_ID = 3  # [MASK] token ID

torch.manual_seed(42)
model = MiniBERT(VOCAB_SIZE, d_model=64, num_heads=4, num_layers=2)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Simulated training sentences
sentences = [
    [5, 8, 10, 6, 9, 4],   # I like cats they very cute
    [5, 12, 13, 14, 4],    # I hate rainy days
    [7, 8, 11, 6, 15, 4],  # He likes dogs they loyal
    [5, 16, 17, 18, 4],    # I am learning programming
    [7, 16, 19, 20, 4],    # He is listening to music
    [5, 21, 22, 9, 23, 4], # I think math very hard
]

# MLM data construction function
def create_mlm_batch(sentences, mask_prob=0.15):
    """
    Same as real BERT:
    - 15% of tokens are selected
    - Of those, 80% replaced with [MASK]
    - 10% replaced with a random token
    - 10% left unchanged (but still predicted)
    """
    # Padding
    max_len = max(len(s) for s in sentences)
    batch_size = len(sentences)
    input_ids = torch.zeros(batch_size, max_len, dtype=torch.long)
    labels = torch.full((batch_size, max_len), -100, dtype=torch.long)  # -100 = ignore

    for i, sent in enumerate(sentences):
        input_ids[i, :len(sent)] = torch.tensor(sent)

        # Select positions to mask (15%), but don't mask first and last
        maskable = list(range(1, len(sent) - 1))
        num_mask = max(1, int(len(maskable) * mask_prob))
        mask_positions = torch.randperm(len(maskable))[:num_mask]

        for pos_idx in mask_positions:
            pos = maskable[pos_idx]
            labels[i, pos] = sent[pos]  # record original token
            rand = torch.rand(1).item()
            if rand < 0.8:
                input_ids[i, pos] = MASK_ID  # 80%: replace with [MASK]
            elif rand < 0.9:
                input_ids[i, pos] = torch.randint(5, 25, (1,)).item()  # 10%: random
            # 10%: keep unchanged

    return input_ids, labels

# Create batch
input_ids, labels = create_mlm_batch(sentences)

print("=== MLM Training Data Examples ===")
print(f"MASK_ID = {MASK_ID}")
print()
for i in range(len(sentences)):
    print(f"Sentence {i+1}:")
    print(f"  Original: {sentences[i]}")
    print(f"  Input:    {input_ids[i].tolist()}")
    label_row = labels[i].tolist()
    print(f"  Labels:   {[('MASK→'+str(l) if l != -100 else 'ignore') for l in label_row]}")
    print()

# ============================================================
# Training
# ============================================================
print("=== Training MiniBERT (MLM) ===")
NUM_EPOCHS = 200
losses = []

model.train()
for epoch in range(NUM_EPOCHS):
    optimizer.zero_grad()
    logits = model(input_ids)  # (B, S, V)
    loss = F.cross_entropy(
        logits.view(-1, VOCAB_SIZE),
        labels.view(-1),
        ignore_index=-100  # ignore non-masked positions
    )
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

    if epoch % 20 == 0 or epoch == NUM_EPOCHS - 1:
        print(f"Epoch {epoch:3d} | Loss: {loss.item():.4f}")

print(f"\nInitial Loss: {losses[0]:.4f} → Final Loss: {losses[-1]:.4f}")

In [ ]:
# ============================================================
# Test: MLM prediction results
# ============================================================
import torch
test_sentence = torch.tensor([[5, MASK_ID, 10, 6, MASK_ID, 4]])  # I [MASK] cats they [MASK] cute

model.eval()
with torch.no_grad():
    logits = model(test_sentence)
    predictions = logits.argmax(dim=-1)

print("=== MLM Prediction Results ===")
print(f"Input:      {test_sentence.tolist()[0]}")
print(f"            ['I', '[MASK]', 'cats', 'they', '[MASK]', 'cute']")
print(f"Predicted:  {predictions.tolist()[0]}")
print(f"            ['I', '?', 'cats', 'they', '?', 'cute']")
print(f"\nPosition 1 (predicted) = {predictions[0, 1].item()} (expected 8='like')")
print(f"Position 4 (predicted) = {predictions[0, 4].item()} (expected 9='very')")

# Visualize loss
import matplotlib.pyplot as plt
plt.figure(figsize=(6, 3))
plt.plot(losses, 'b-', linewidth=1)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('MiniBERT MLM Training Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. BERT's Fine-tuning Paradigm

After BERT is pre-trained, different downstream tasks only need a different classification head attached on top. The key question is: **where is the classification head attached?**

The pre-trained BERT body is a "general-purpose text encoder" — given a piece of text as input, it outputs a hidden state for each token. Different tasks require information from different positions:

| Task | What information is needed | Where to attach the classification head |
|------|---------------------------|----------------------------------------|
| **Single-sentence classification** (sentiment analysis) | Semantics of the whole sentence | `[CLS]` hidden → classification layer |
| **Sentence-pair classification** (NLI, similarity) | Relationship between two sentences | `[CLS]` hidden → classification layer |
| **Sequence labeling** (NER) | Category of each word | **Each token's** hidden → individual classification layers |
| **Question answering** (SQuAD) | Start and end positions of the answer | Each token's hidden → start/end classification layers |

Why do both single-sentence and sentence-pair classification use `[CLS]`? Because `[CLS]` sits at the very beginning of BERT's input. After passing through multiple layers of bidirectional attention, information from the entire sentence is aggregated into `[CLS]`'s hidden state — it serves as a "whole-sentence summary". Passing it through a classification layer is sufficient for sentence-level judgments.

Sequence labeling tasks (e.g. NER: labeling "Apple released a new phone" as `[ORG, O, O, O]`) require a category for each token individually, so we cannot just use `[CLS]` — we need to classify each position's hidden state separately.

Core insight: **The BERT body usually stays frozen (or receives only minor fine-tuning), and only the final "head" is swapped.** This is a different paradigm from GPT's prompt engineering — BERT adapts to tasks by swapping heads, while GPT adapts to tasks by swapping prompts.

In [ ]:
import torch.nn as nn
# ============================================================
# Demo: Using MiniBERT for sentence classification
# ============================================================
import torch

class MiniBERTForClassification(nn.Module):
    """Attach a classification head on top of MiniBERT"""
    def __init__(self, bert, num_classes):
        super().__init__()
        self.bert = bert
        self.classifier = nn.Linear(bert.d_model, num_classes)

    def forward(self, x):
        hidden = self.bert.encode(x)  # (B, S, D)
        cls_output = hidden[:, 0, :]  # take [CLS] output
        return self.classifier(cls_output)  # → (B, num_classes)

# Demo
VOCAB_SIZE = 50
bert_base = MiniBERT(VOCAB_SIZE, d_model=64, num_heads=4, num_layers=2)
clf_model = MiniBERTForClassification(bert_base, num_classes=2)  # binary: positive/negative

test_input = torch.randint(0, VOCAB_SIZE, (1, 10))
output = clf_model(test_input)
print(f"Input shape: {test_input.shape}")
print(f"Output shape: {output.shape}  ← (batch=1, classes=2)")
print(f"Output logits: {output.tolist()}")
print(f"\nThis is the complete pipeline for BERT sentiment analysis:")
print(f"  Input sentence → BERT encoding → take [CLS] → classification layer → positive/negative")

## 6. Loading Real BERT Demo

Above we implemented MiniBERT from scratch, understanding the underlying mechanisms of the Encoder-only architecture and MLM training. Now let's load a real BERT model using HuggingFace transformers and see how the industrial-scale implementation maps to our own version.

The key things to notice: real BERT scales up in three dimensions —
- **Scale**: BERT-base has 12 Transformer blocks, 768-dimensional hidden size, 12 attention heads, totaling about 110M parameters
- **Vocabulary**: Uses a 30,522-token WordPiece vocabulary, far larger than the dozens of tokens in our demo
- **Pre-training data**: BooksCorpus (800M words) + English Wikipedia (2,500M words)

But the core architecture is identical to our MiniBERT: both are Encoder-only, both use bidirectional attention, and both learn contextual understanding through MLM pre-training.

In [ ]:
# ============================================================
# Load real BERT using transformers
# ============================================================
import torch
try:
    from transformers import AutoTokenizer, AutoModel

    print("Loading bert-base-chinese...")
    tokenizer = AutoTokenizer.from_pretrained("bert-base-chinese")
    model = AutoModel.from_pretrained("bert-base-chinese")

    print(f"BERT parameter count: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M")
    print(f"Vocabulary size: {len(tokenizer)}")
    print()

    # Test: look at BERT's attention weights
    sentences = [
        "I love China",
        "The weather is nice today",
    ]

    inputs = tokenizer(sentences, padding=True, return_tensors="pt")
    print(f"Input IDs shape: {inputs['input_ids'].shape}")
    print(f"Attention mask: {inputs['attention_mask']}")
    print()

    # Forward pass
    with torch.no_grad():
        outputs = model(**inputs, output_attentions=True)

    print(f"Output last_hidden_state shape: {outputs.last_hidden_state.shape}")
    print(f"  → (batch, seq_len, hidden_dim=768)")
    print()

    # Look at the last layer's attention (first head)
    last_layer_attn = outputs.attentions[-1]  # (batch, num_heads, seq_len, seq_len)
    print(f"Last layer attention shape: {last_layer_attn.shape}")
    print(f"  → (batch, 12 heads, seq_len, seq_len)")
    print()

    # Show what each token in the first sentence attends to
    tokens1 = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    print(f"Sentence 1 tokens: {tokens1}")
    print(f"\nAttention distribution for each token (head 0):")
    for i, tok in enumerate(tokens1):
        attn_weights = last_layer_attn[0, 0, i]  # sample 0, head 0, position i
        top2 = attn_weights.topk(2)
        print(f"  '{tok}' most attends to: ", end="")
        for j, (idx, w) in enumerate(zip(top2.indices, top2.values)):
            print(f"'{tokens1[idx]}'({w:.2f})", end="  ")
        print()

except ImportError:
    print("transformers library not installed. Run: pip install transformers")
except Exception as e:
    print(f"Error loading BERT: {e}")
    print("(This is normal — if the network is unavailable or the model is too large, refer to the MiniBERT demo above)")

## 7. BERT vs. GPT Comparison

| | BERT (Encoder-Only) | GPT (Decoder-Only) |
|------|------|------|
| **Core task** | Understanding — "What does this sentence mean?" | Generation — "What should come next?" |
| **Pre-training** | MLM (mask and fill, 15% mask) | Autoregressive (predict next token) |
| **Attention** | Bidirectional (sees whole sentence) | Unidirectional / causal (sees only preceding) |
| **Input representation** | Token + Segment + Position | Token + Position (no Segment) |
| **Output** | Hidden state per position | Logits per position (but generation uses only the last) |
| **How to use** | Attach classification head and fine-tune | Change prompt / dialogue format |
| **Representative models** | BERT, RoBERTa, DeBERTa | GPT-3/4, LLaMA, Qwen, DeepSeek |
| **Why GPT won** | BERT cannot generate, and its emergent abilities at scale are weaker than GPT | GPT can also do understanding via in-context learning |

BERT and GPT are two applications of the same Transformer paper — one uses the Encoder for understanding, the other uses the Decoder for generation. GPT won the scaling race, but BERT's ideas have not disappeared:

- RoBERTa removed BERT's NSP task (finding it unhelpful), using only MLM with more data and longer training, achieving significantly better results
- DistilBERT compressed BERT by 40% through knowledge distillation, speeding it up by 60%, while retaining 97% of performance
- DeBERTa improved the Attention mechanism (decoupling relative position from content), at one point surpassing human baselines on SuperGLUE
- MLM's "cover some parts and ask the model to guess" strategy has been widely adopted by T5, BART, and even modern multimodal models

**One-sentence summary**: BERT uses bidirectional Attention for MLM, excelling at understanding but not at generation. GPT uses unidirectional Attention for autoregressive prediction, excelling at generation and, as scale grows, developing emergent understanding abilities. BERT's MLM approach and bidirectional encoding design continue to influence the entire NLP field.

## Summary

What we learned in this section:

- [ ] Encoder-only models (like BERT) use bidirectional Attention, allowing them to see the complete input sequence
- [ ] BERT's pre-training task is MLM (mask and fill), not autoregressive "predict the next token"
- [ ] BERT's input is the sum of three Embeddings: Token + Segment + Position
- [ ] During fine-tuning, a classification head is attached on top of BERT; the [CLS] token's representation is used for downstream tasks
- [ ] GPT won the scaling race, but BERT's MLM approach and bidirectional encoding continue to influence the NLP field

## Exercises

> You can ask AI for help explaining the approach, but it is not recommended to ask AI to "complete this exercise outright."

**Exercise 1: MLM mask ratio**

BERT's MLM task defaults to masking 15% of tokens. This ratio was not chosen at random.

**Hint**: Think about what happens when too many or too few tokens are masked.